In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.animation import FFMpegWriter  # ou PillowWriter
import os
import json
from matplotlib import gridspec
import imageio.v2 as imageio
import matplotlib as mpl
from PIL import Image, ImageDraw, ImageFont



## GROWTH NETWORK - 2D

In [3]:

# ---------------- Parâmetros base ----------------
dim = 2
L   = 1000          # usado no path; L real vem do shape no npz
nc  = 2
rho = 1/nc
k   = 1.0e-05
Nt  = 200

# ---------------- Paths ----------------
path_json = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/P0_0.10_p0_1.00_seed_1.json"
path_npz  = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/P0_0.10_p0_1.00_seed_1.npz"

# mesmas cores do crescimento da rede
colors_used = {
    1: (0.9, 0.1, 0.1),  # vermelho
    2: (0.1, 0.1, 0.9),  # azul
}

out_dir = os.path.dirname(path_json)
out_gif = os.path.join(out_dir, "P0_0.10_p0_1.00_seed_1_props.gif")

# ============================================================
# 1) Carrega o NPZ para pegar frame_times (mesmos da rede)
# ============================================================
data_npz = np.load(path_npz)
shape      = tuple(data_npz["shape"])
M_flat     = data_npz["data"]
num_colors = int(data_npz["num_colors"])

print("NPZ shape salvo:", shape)
print("num_colors:", num_colors)

assert np.prod(shape) == M_flat.size, "shape x size inconsistente"

M = M_flat.reshape(shape)
print("M.shape real:", M.shape)

# Decodificação M = c*1e6 + t
M = M.astype(np.int32, copy=False)
color_mat = (M // 1_000_000).astype(np.int16)   # 1 ou 2
time_raw  = (M %  1_000_000).astype(np.int32)   # 0..tmax ou 999999

print("cores não nulas (NPZ):", np.unique(color_mat[color_mat > 0]))

T_SENTINELA = 999_999
mask_valid_time = (time_raw < T_SENTINELA) & (color_mat > 0)
print("N sítios com tempo válido:", mask_valid_time.sum())

t_vals = np.unique(time_raw[mask_valid_time])
t_min_phys, t_max_phys = int(t_vals[0]), int(t_vals[-1])
print(f"t físico min: {t_min_phys}, max: {t_max_phys}, qtd t distintos: {len(t_vals)}")

# Lista de tempos (frames) — deve ser a MESMA usada no GIF da rede
frame_stride = 1        # se mudar aqui, mude também no gif da rede
frame_times  = t_vals[::frame_stride]
n_frames     = frame_times.size
print("Número de frames:", n_frames)

# ============================================================
# 2) Carrega JSON e prepara curvas p(t), N(t)
# ============================================================
with open(path_json, "r") as f:
    data = json.load(f)

results = data["results"]

curves = {}
for v in results.values():
    d = v["data"]
    c = d["color"]
    if c in (1, 2):
        curves[c] = {
            "time": np.asarray(d["time"], dtype=float),
            "pt":   np.asarray(d["pt"],   dtype=float),
            "nt":   np.asarray(d["nt"],   dtype=float),
        }

print("cores presentes no JSON:", list(curves.keys()))

# assume que as duas cores têm o mesmo vetor de tempo
t_arr = curves[1]["time"]  # vetor de tempo do JSON (0..tmax)
t_min, t_max = t_arr[0], t_arr[-1]

# limites globais em Y (pt e Nt)
pt_min = min(curves[1]["pt"].min(), curves[2]["pt"].min())
pt_max = max(curves[1]["pt"].max(), curves[2]["pt"].max())
nt_min = min(curves[1]["nt"].min(), curves[2]["nt"].min())
nt_max = max(curves[1]["nt"].max(), curves[2]["nt"].max())

pt_margin = 0.05 * (pt_max - pt_min)
nt_margin = 0.05 * (nt_max - nt_min)

thickness_axes = 1.2
fs_ticks = 12
pc_reff = 0.5

# ============================================================
# 3) Cria figura base
# ============================================================
plt.close("all")
fig = plt.figure(figsize=(6, 6), dpi=150)
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1], hspace=0.15)

ax_pt = fig.add_subplot(gs[0])
ax_nt = fig.add_subplot(gs[1], sharex=ax_pt)

ax_pt.set_ylabel(r"$p(t)$", fontsize=15)
ax_nt.set_ylabel(r"$N(t)$", fontsize=15)
ax_nt.set_xlabel("$t$", fontsize=15)

ax_pt.set_xlim(t_min_phys, t_max_phys)   # usa limites físicos (iguais ao gif da rede)
ax_nt.set_xlim(t_min_phys, t_max_phys)
ax_pt.set_ylim(pt_min - pt_margin, pt_max + pt_margin)
ax_nt.set_ylim(nt_min - nt_margin, nt_max + nt_margin)

ax_pt.tick_params(axis='both', which='major',
                  length=4, width=thickness_axes, direction='in', labelsize=fs_ticks)
ax_nt.tick_params(axis='both', which='major',
                  length=4, width=thickness_axes, direction='in', labelsize=fs_ticks)

# esconde labels de x no eixo superior
ax_pt.tick_params(axis='x', which='both', labelbottom=False)

ax_pt.axhline(y=pc_reff,xmin=0,xmax=1, ls='--',color='k',label=f"$p_c = {pc_reff}$")
ax_nt.axhline(y=Nt, xmin=0, xmax=1, ls='--',color='k',label=f"$N_T = {Nt}$")

lines_pt = {}
lines_nt = {}
for c in (1, 2):
    col = colors_used[c]
    (lp,) = ax_pt.plot([], [], color=col, lw=1.5, label=f"Color {c}")
    (ln,) = ax_nt.plot([], [], color=col, lw=1.5, label=f"Color {c}")
    lines_pt[c] = lp
    lines_nt[c] = ln

ax_pt.legend(loc="upper right", fontsize=10)
ax_nt.legend(loc="upper right", fontsize=10)

fig.tight_layout()

# ============================================================
# 4) Gera GIF (sem loop infinito) sincronizado com frame_times
# ============================================================

# frame_duration compatível com fps=20 (como no PillowWriter): 1/20 = 0.05 s
frame_duration = 1.0 / 20.0   # ~20 fps

with imageio.get_writer(out_gif, mode="I", duration=frame_duration, loop=1) as writer:
    for i, t in enumerate(frame_times):
        # máscara de pontos com tempo <= t (usando tempo do JSON)
        mask_t = t_arr <= t

        for c in (1, 2):
            lines_pt[c].set_data(t_arr[mask_t], curves[c]["pt"][mask_t])
            lines_nt[c].set_data(t_arr[mask_t], curves[c]["nt"][mask_t])

        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        writer.append_data(frame)

        if (i % 50) == 0 or i == len(frame_times) - 1:
            print(f"[{i+1:5d}/{len(frame_times)}]  t = {t:4d}/{t_max_phys}")

plt.close(fig)
print("GIF salvo em:", out_gif)


NPZ shape salvo: (1000, 1000)
num_colors: 2
M.shape real: (1000, 1000)
cores não nulas (NPZ): [1 2]
N sítios com tempo válido: 779007
t físico min: 0, max: 1704, qtd t distintos: 1705
Número de frames: 1705
cores presentes no JSON: [1, 2]
[    1/1705]  t =    0/1704


/tmp/ipykernel_30060/1269990696.py:139: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[   51/1705]  t =   50/1704
[  101/1705]  t =  100/1704
[  151/1705]  t =  150/1704
[  201/1705]  t =  200/1704
[  251/1705]  t =  250/1704
[  301/1705]  t =  300/1704
[  351/1705]  t =  350/1704
[  401/1705]  t =  400/1704
[  451/1705]  t =  450/1704
[  501/1705]  t =  500/1704
[  551/1705]  t =  550/1704
[  601/1705]  t =  600/1704
[  651/1705]  t =  650/1704
[  701/1705]  t =  700/1704
[  751/1705]  t =  750/1704
[  801/1705]  t =  800/1704
[  851/1705]  t =  850/1704
[  901/1705]  t =  900/1704
[  951/1705]  t =  950/1704
[ 1001/1705]  t = 1000/1704
[ 1051/1705]  t = 1050/1704
[ 1101/1705]  t = 1100/1704
[ 1151/1705]  t = 1150/1704
[ 1201/1705]  t = 1200/1704
[ 1251/1705]  t = 1250/1704
[ 1301/1705]  t = 1300/1704
[ 1351/1705]  t = 1350/1704
[ 1401/1705]  t = 1400/1704
[ 1451/1705]  t = 1450/1704
[ 1501/1705]  t = 1500/1704
[ 1551/1705]  t = 1550/1704
[ 1601/1705]  t = 1600/1704
[ 1651/1705]  t = 1650/1704
[ 1701/1705]  t = 1700/1704
[ 1705/1705]  t = 1704/1704
GIF salvo em: ../ani

## GROWTH PROPERTIES

In [3]:
plt.style.use('properties.mplstyle')

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
import imageio.v2 as imageio
import matplotlib as mpl

# ---------------- Parâmetros base ----------------
dim = 2
L   = 1000          # usado no path; L real vem do shape no npz
nc  = 2
rho = 1/nc
k   = 1.0e-05
Nt  = 200

# ---------------- Paths ----------------
path_json = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/P0_0.10_p0_1.00_seed_1.json"
path_npz  = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/P0_0.10_p0_1.00_seed_1.npz"

# mesmas cores do crescimento da rede
colors_used = {
    1: (0.9, 0.1, 0.1),  # vermelho
    2: (0.1, 0.1, 0.9),  # azul
}

out_dir = os.path.dirname(path_json)
out_gif = os.path.join(out_dir, "P0_0.10_p0_1.00_seed_1_props.gif")

# ============================================================
# 1) Carrega o NPZ para pegar frame_times (mesmos da rede)
# ============================================================
data_npz = np.load(path_npz)
shape      = tuple(data_npz["shape"])
M_flat     = data_npz["data"]
num_colors = int(data_npz["num_colors"])

print("NPZ shape salvo:", shape)
print("num_colors:", num_colors)

assert np.prod(shape) == M_flat.size, "shape x size inconsistente"

M = M_flat.reshape(shape)
print("M.shape real:", M.shape)

# Decodificação M = c*1e6 + t
M = M.astype(np.int32, copy=False)
color_mat = (M // 1_000_000).astype(np.int16)   # 1 ou 2
time_raw  = (M %  1_000_000).astype(np.int32)   # 0..tmax ou 999999

print("cores não nulas (NPZ):", np.unique(color_mat[color_mat > 0]))

T_SENTINELA = 999_999
mask_valid_time = (time_raw < T_SENTINELA) & (color_mat > 0)
print("N sítios com tempo válido:", mask_valid_time.sum())

t_vals = np.unique(time_raw[mask_valid_time])
t_min_phys, t_max_phys = int(t_vals[0]), int(t_vals[-1])
print(f"t físico min: {t_min_phys}, max: {t_max_phys}, qtd t distintos: {len(t_vals)}")

# Lista de tempos (frames) — deve ser a MESMA usada no GIF da rede
frame_stride = 1        # se mudar aqui, mude também no gif da rede
frame_times  = t_vals[::frame_stride]
n_frames     = frame_times.size
print("Número de frames:", n_frames)

# ============================================================
# 2) Carrega JSON e prepara curvas p(t), N(t)
# ============================================================
with open(path_json, "r") as f:
    data = json.load(f)

results = data["results"]

curves = {}
for v in results.values():
    d = v["data"]
    c = d["color"]
    if c in (1, 2):
        curves[c] = {
            "time": np.asarray(d["time"], dtype=float),
            "pt":   np.asarray(d["pt"],   dtype=float),
            "nt":   np.asarray(d["nt"],   dtype=float),
        }

print("cores presentes no JSON:", list(curves.keys()))

# assume que as duas cores têm o mesmo vetor de tempo
t_arr = curves[1]["time"]  # vetor de tempo do JSON (0..tmax)
t_min, t_max = t_arr[0], t_arr[-1]

# limites globais em Y (pt e Nt)
pt_min = min(curves[1]["pt"].min(), curves[2]["pt"].min())
pt_max = max(curves[1]["pt"].max(), curves[2]["pt"].max())
nt_min = min(curves[1]["nt"].min(), curves[2]["nt"].min())
nt_max = max(curves[1]["nt"].max(), curves[2]["nt"].max())

pt_margin = 0.05 * (pt_max - pt_min)
nt_margin = 0.05 * (nt_max - nt_min)

thickness_axes = 1.2
fs_ticks = 12

# ============================================================
# 3) Cria figura base
# ============================================================
plt.close("all")
fig = plt.figure(figsize=(6, 6), dpi=150)
gs = gridspec.GridSpec(2, 1, height_ratios=[1, 1], hspace=0.15)

ax_pt = fig.add_subplot(gs[0])
ax_nt = fig.add_subplot(gs[1], sharex=ax_pt)

ax_pt.set_ylabel(r"$p(t)$", fontsize=15)
ax_nt.set_ylabel(r"$N(t)$", fontsize=15)
ax_nt.set_xlabel("$t$", fontsize=15)

ax_pt.set_xlim(t_min_phys, t_max_phys)   # usa limites físicos (iguais ao gif da rede)
ax_nt.set_xlim(t_min_phys, t_max_phys)
ax_pt.set_ylim(pt_min - pt_margin, pt_max + pt_margin)
ax_nt.set_ylim(nt_min - nt_margin, nt_max + nt_margin)

pc_reff = 0.50
ax_pt.axhline(y=pc_reff, xmin=0, xmax=1, ls='--', color='k',
              label=f"$p_c = {pc_reff}$")
ax_nt.axhline(y=Nt, xmin=0, xmax=1, ls='--', color='k',
              label=f"$N_T = {Nt}$")

ax_pt.tick_params(axis='both', which='major',
                  length=4, width=thickness_axes, direction='in', labelsize=fs_ticks)
ax_nt.tick_params(axis='both', which='major',
                  length=4, width=thickness_axes, direction='in', labelsize=fs_ticks)

# esconde labels de x no eixo superior
ax_pt.tick_params(axis='x', which='both', labelbottom=False)

lines_pt = {}
lines_nt = {}
for c in (1, 2):
    col = colors_used[c]
    (lp,) = ax_pt.plot([], [], color=col, lw=1.5, label=f"Color {c}")
    (ln,) = ax_nt.plot([], [], color=col, lw=1.5, label=f"Color {c}")
    lines_pt[c] = lp
    lines_nt[c] = ln

ax_pt.legend(loc="upper right", fontsize=10)
ax_nt.legend(loc="upper right", fontsize=10)

fig.tight_layout()

# ============================================================
# 4) Gera GIF (sem loop infinito) sincronizado com frame_times
# ============================================================

# frame_duration compatível com fps=20 (como no PillowWriter): 1/20 = 0.05 s
frame_duration = 1.0 / 20.0   # ~20 fps

with imageio.get_writer(out_gif, mode="I", duration=frame_duration, loop=1) as writer:
    for i, t in enumerate(frame_times):
        # máscara de pontos com tempo <= t (usando tempo do JSON)
        mask_t = t_arr <= t

        for c in (1, 2):
            lines_pt[c].set_data(t_arr[mask_t], curves[c]["pt"][mask_t])
            lines_nt[c].set_data(t_arr[mask_t], curves[c]["nt"][mask_t])

        fig.canvas.draw()
        frame = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (3,))
        writer.append_data(frame)

        if (i % 50) == 0 or i == len(frame_times) - 1:
            print(f"[{i+1:5d}/{len(frame_times)}]  t = {t:4d}/{t_max_phys}")

plt.close(fig)
print("GIF salvo em:", out_gif)


NPZ shape salvo: (1000, 1000)
num_colors: 2
M.shape real: (1000, 1000)
cores não nulas (NPZ): [1 2]
N sítios com tempo válido: 779007
t físico min: 0, max: 1704, qtd t distintos: 1705
Número de frames: 1705
cores presentes no JSON: [1, 2]
[    1/1705]  t =    0/1704


/tmp/ipykernel_30060/3950391987.py:143: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[   51/1705]  t =   50/1704
[  101/1705]  t =  100/1704
[  151/1705]  t =  150/1704
[  201/1705]  t =  200/1704
[  251/1705]  t =  250/1704
[  301/1705]  t =  300/1704
[  351/1705]  t =  350/1704
[  401/1705]  t =  400/1704
[  451/1705]  t =  450/1704
[  501/1705]  t =  500/1704
[  551/1705]  t =  550/1704
[  601/1705]  t =  600/1704
[  651/1705]  t =  650/1704
[  701/1705]  t =  700/1704
[  751/1705]  t =  750/1704
[  801/1705]  t =  800/1704
[  851/1705]  t =  850/1704
[  901/1705]  t =  900/1704
[  951/1705]  t =  950/1704
[ 1001/1705]  t = 1000/1704
[ 1051/1705]  t = 1050/1704
[ 1101/1705]  t = 1100/1704
[ 1151/1705]  t = 1150/1704
[ 1201/1705]  t = 1200/1704
[ 1251/1705]  t = 1250/1704
[ 1301/1705]  t = 1300/1704
[ 1351/1705]  t = 1350/1704
[ 1401/1705]  t = 1400/1704
[ 1451/1705]  t = 1450/1704
[ 1501/1705]  t = 1500/1704
[ 1551/1705]  t = 1550/1704
[ 1601/1705]  t = 1600/1704
[ 1651/1705]  t = 1650/1704
[ 1701/1705]  t = 1700/1704
[ 1705/1705]  t = 1704/1704
GIF salvo em: ../ani

## COMBINE GIFS

In [5]:
import os
import numpy as np
import imageio.v2 as imageio
from PIL import Image
import matplotlib.pyplot as plt

# ---------------- Parâmetros base ----------------
dim = 2
L   = 1000          # usado no path; L real vem do shape no npz
nc  = 2
rho = 1/nc
k   = 1.0e-05
Nt  = 200


# Diretório onde estão os GIFs
base_dir   = f"../animation/{dim}D_L{L}_nc{nc}_rho{rho:.2f}_k{k:.1e}_Nt{Nt}/"

left_gif   = os.path.join(base_dir, f"L{L}_seed1_anim.gif")
right_gif  = os.path.join(base_dir, "P0_0.10_p0_1.00_seed_1_props.gif")
out_gif    = os.path.join(base_dir, "combined.gif")

# Título em "LaTeX" (mathtext do Matplotlib)
title_text = (
    r"Network with $L = %d$, $n_c = %d$, $\rho = %.2f$, "
    r"$k = %.1e$, $N_T = %d$"
) % (L, nc, rho, k, Nt)

# --------------------------------------------------
# 1) Abre os readers em modo streaming
# --------------------------------------------------
reader_left  = imageio.get_reader(left_gif, mode="I")
reader_right = imageio.get_reader(right_gif, mode="I")

n_left  = reader_left.get_length()
n_right = reader_right.get_length()
n_frames = min(n_left, n_right)

print(f"Frames left: {n_left}, right: {n_right}, usados: {n_frames}")

# --------------------------------------------------
# 2) Descobre tamanhos
# --------------------------------------------------
first_left  = reader_left.get_data(0)
first_right = reader_right.get_data(0)

imL0 = Image.fromarray(first_left)
imR0 = Image.fromarray(first_right)

wL, hL = imL0.size
wR, hR = imR0.size

# Redimensiona o da direita para ter a mesma altura do da esquerda
target_h = hL
scale_R  = target_h / hR
new_wR   = int(wR * scale_R)

combined_w = wL + new_wR

# --------------------------------------------------
# 3) Gera uma faixa de título com Matplotlib (reutilizada em todos os frames)
# --------------------------------------------------
def make_title_strip(width_px, height_px, text):
    dpi = 100
    fig = plt.figure(figsize=(width_px / dpi, height_px / dpi), dpi=dpi)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")

    # fontsize maior para destacar bem
    ax.text(
        0.5, 0.5, text,
        ha="center", va="center",
        fontsize=30
    )

    fig.canvas.draw()
    w, h = fig.canvas.get_width_height()
    img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    img = img.reshape((h, w, 3))
    plt.close(fig)
    return Image.fromarray(img)

# altura aproximada do título
title_nominal_h = 80
title_img = make_title_strip(combined_w, title_nominal_h, title_text)
title_h = title_img.height

combined_h = title_h + target_h

# --------------------------------------------------
# 4) Cria GIF combinado em streaming
# --------------------------------------------------

# Mesmo "fps" do PillowWriter da rede (20 fps => 0.05 s por frame)
frame_duration = 1.0 / 20.0

with imageio.get_writer(out_gif, mode="I", duration=frame_duration, loop=1) as writer:
    for i in range(n_frames):
        # Lê frame i de cada GIF
        frameL = reader_left.get_data(i)
        frameR = reader_right.get_data(i)

        imL = Image.fromarray(frameL)
        imR = Image.fromarray(frameR)

        # Redimensiona direita
        imR = imR.resize((new_wR, target_h), resample=Image.BILINEAR)

        # Canvas combinado (fundo branco)
        combined = Image.new("RGB", (combined_w, combined_h), color=(255, 255, 255))

        # Cola o título (já com LaTeX renderizado pelo Matplotlib)
        combined.paste(title_img, (0, 0))

        # Cola os dois GIFs abaixo do título
        combined.paste(imL, (0,      title_h))
        combined.paste(imR, (wL,     title_h))

        # Adiciona frame ao GIF final
        writer.append_data(np.array(combined))

        if (i % 50) == 0 or i == n_frames - 1:
            print(f"Frame {i+1}/{n_frames}")

# Fecha os readers
reader_left.close()
reader_right.close()

print("GIF combinado salvo em:", out_gif)

Frames left: 1705, right: 1704, usados: 1704
Frame 1/1704
Frame 51/1704
Frame 101/1704
Frame 151/1704
Frame 201/1704
Frame 251/1704
Frame 301/1704
Frame 351/1704
Frame 401/1704
Frame 451/1704
Frame 501/1704
Frame 551/1704
Frame 601/1704
Frame 651/1704
Frame 701/1704
Frame 751/1704
Frame 801/1704
Frame 851/1704
Frame 901/1704
Frame 951/1704
Frame 1001/1704
Frame 1051/1704
Frame 1101/1704
Frame 1151/1704
Frame 1201/1704
Frame 1251/1704
Frame 1301/1704
Frame 1351/1704
Frame 1401/1704
Frame 1451/1704
Frame 1501/1704
Frame 1551/1704
Frame 1601/1704
Frame 1651/1704
Frame 1701/1704
Frame 1704/1704
GIF combinado salvo em: ../animation/2D_L1000_nc2_rho0.50_k1.0e-05_Nt200/combined.gif
